# Filtering (stream compaction) on the GPU: chapter 12

A **filter** keeps the elements of an array that satisfy a condition and packs them at the front of the output:

This notebook runs the **complete programs**: every kernel of the chapter on large inputs, compared side by side with timings, and the **bike-sharing examples**, which apply the same pattern to another domain to give you ideas for your project. Do the step-by-step notebook (`colab_1_step_by_step.ipynb`) first. Timings are only meaningful on the GPU.

The full theory, the exercises and their solutions are in `ec2/10_filter/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_idle_bikes.cu`

many filters in one 2-D grid: docked, charged bikes within a radius of each rider request, output grouped by request

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_idle_bikes.cu
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#include "cuda_compat.h"

constexpr int BK_THREADS = 256;
constexpr int BK_SECTION = 2 * BK_THREADS;
constexpr int F_THREADS = 256;
constexpr int F_COARSEN = 8;
constexpr int F_SECTION = F_THREADS * F_COARSEN;

constexpr int OFFSET_THREADS = 256;

#ifndef CPU_BIKES
#define CPU_BIKES 12000
#endif
#ifndef CPU_REQUESTS
#define CPU_REQUESTS 16
#endif
constexpr int DEFAULT_BIKES = BY_BACKEND(200000, CPU_BIKES);
constexpr int DEFAULT_REQUESTS = BY_BACKEND(1024, CPU_REQUESTS);
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr uint64_t SNAPSHOT_SEED = 99;

constexpr int CITY_SIDE_M = 6000;
constexpr int STATIONS = 48;
constexpr int SEARCH_RADIUS_M = 600;
constexpr int MIN_BATTERY_PERCENT = 30;
constexpr int FULL_BATTERY_PERCENT = 100;
constexpr int EXPECTED_CANDIDATES_PER_REQUEST = 128;
constexpr int REQUESTS_SHOWN = 6;

constexpr int DOCKED_WEIGHT = 45;
constexpr int RIDING_WEIGHT = 45;
constexpr int IN_REPAIR_WEIGHT = 10;
constexpr int NOT_DOCKED = -1;

enum class BikeState : int { Docked, Riding, InRepair };

__host__ __device__ inline bool isReadyBike(
    BikeState state, int battery, int x, int y, int qx, int qy, int radius) {
  const int dx = x - qx;
  const int dy = y - qy;
  return state == BikeState::Docked && battery >= MIN_BATTERY_PERCENT &&
         dx * dx + dy * dy <= radius * radius;
}

__global__ void brent_kung_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
  }

  __syncthreads();
  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
  if (t == 0) {
    sums[blockIdx.x] = XY[BK_SECTION - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__device__ void kogge_stone_in_shared(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < F_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ int count_own_flags(const int* flag, int first) {
  int count = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    count += flag[first + k];
  }
  return count;
}

__device__ void pack_own_flagged_indices(
    const int* flag, int first, int base, int pos, int* stage) {
  for (int k = 0; k < F_COARSEN; ++k) {
    if (flag[first + k]) {
      stage[pos++] = base + first + k;
    }
  }
}

__device__ void copy_out_coalesced(const int* stage, int count, int* out) {
  for (int k = threadIdx.x; k < count; k += F_THREADS) {
    out[k] = stage[k];
  }
}

__global__ void count_candidates(const BikeState* state,
                                 const int* battery,
                                 const int* x,
                                 const int* y,
                                 const int* qx,
                                 const int* qy,
                                 int* counts,
                                 int nBikes,
                                 int radius) {
  __shared__ int blockCount;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const int q = blockIdx.y;
  const int rx = qx[q];
  const int ry = qy[q];
  int c = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    const int i = blockIdx.x * F_SECTION + k * F_THREADS + threadIdx.x;
    if (i < nBikes && isReadyBike(state[i], battery[i], x[i], y[i], rx, ry, radius)) {
      ++c;
    }
  }
  atomicAdd(&blockCount, c);
  __syncthreads();

  if (threadIdx.x == 0) {
    counts[q * gridDim.x + blockIdx.x] = blockCount;
  }
}

__global__ void compact_candidates(const BikeState* state,
                                   const int* battery,
                                   const int* x,
                                   const int* y,
                                   const int* qx,
                                   const int* qy,
                                   const int* scanned,
                                   int* candidates,
                                   int nBikes,
                                   int radius) {
  __shared__ int flag[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  const int t = threadIdx.x;
  const int q = blockIdx.y;
  const int rx = qx[q];
  const int ry = qy[q];
  const int base = blockIdx.x * F_SECTION;
  const int first = t * F_COARSEN;

  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = k * F_THREADS + t;
    const int i = base + j;
    flag[j] = (i < nBikes && isReadyBike(state[i], battery[i], x[i], y[i], rx, ry, radius)) ? 1 : 0;
  }
  __syncthreads();

  part[t] = count_own_flags(flag, first);
  kogge_stone_in_shared(part);
  __syncthreads();

  pack_own_flagged_indices(flag, first, base, exclusive_from_inclusive(part, t), stage);
  __syncthreads();

  const int cell = q * gridDim.x + blockIdx.x;
  const int start = exclusive_from_inclusive(scanned, cell);
  const int blockCount = part[F_THREADS - 1];
  copy_out_coalesced(stage, blockCount, candidates + start);
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int bound) {
    return static_cast<int>(next() % static_cast<uint64_t>(bound));
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

struct Station {
  int x;
  int y;
};

struct FleetSnapshot {
  std::vector<BikeState> state;
  std::vector<int> battery;
  std::vector<int> x;
  std::vector<int> y;
  std::vector<int> dockedAt;
};

struct RiderRequests {
  std::vector<int> x;
  std::vector<int> y;
};

struct CandidateLists {
  std::vector<int> start;
  std::vector<int> bike;
};

struct DeviceData {
  BikeState* state;
  int* battery;
  int* x;
  int* y;
  int* requestX;
  int* requestY;
  int* counts;
  int* scannedCounts;
  int* scratch;
  int* candidates;
};

int blocksFor(int n, int blockSize) {
  return static_cast<int>(cdiv(n, blockSize));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

template <class T>
T* upload(const std::vector<T>& values) {
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(T)));
  CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(T), cudaMemcpyHostToDevice));
  return pointer;
}

std::vector<int> downloadInts(const int* source, int count) {
  std::vector<int> values(count);
  if (count > 0) {
    CHECK(cudaMemcpy(values.data(), source, count * sizeof(int), cudaMemcpyDeviceToHost));
  }
  return values;
}

int copyIntToHost(const int* source) {
  int value = 0;
  CHECK(cudaMemcpy(&value, source, sizeof(int), cudaMemcpyDeviceToHost));
  return value;
}

int scratchIntsForHierarchicalScan(int n) {
  const int blocks = blocksFor(n, BK_SECTION);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks);
}

void scanInclusive(const int* d_in, int* d_out, int n, int* scratch) {
  const int blocks = blocksFor(n, BK_SECTION);
  int* blockSums = scratch;
  launch(brent_kung_scan, dim3(blocks), dim3(BK_THREADS), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanInclusive(blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(add_block_offsets,
           dim3(blocks),
           dim3(OFFSET_THREADS),
           d_out,
           scannedBlockSums,
           n,
           BK_SECTION);
  }
}

struct SearchGrid {
  int bikes;
  int requests;
  int fleetSections;
  int cells;
};

SearchGrid makeSearchGrid(int bikes, int requests) {
  const int fleetSections = blocksFor(bikes, F_SECTION);
  return {bikes, requests, fleetSections, fleetSections * requests};
}

dim3 sectionsByRequestsGrid(const SearchGrid& grid) {
  return dim3(grid.fleetSections, grid.requests);
}

std::vector<Station> makeStations(SplitMix64& rng) {
  std::vector<Station> stations(STATIONS);
  for (Station& station : stations) {
    station.x = rng.below(CITY_SIDE_M);
    station.y = rng.below(CITY_SIDE_M);
  }
  return stations;
}

BikeState pickState(SplitMix64& rng) {
  const int ticket = rng.below(DOCKED_WEIGHT + RIDING_WEIGHT + IN_REPAIR_WEIGHT);
  if (ticket < DOCKED_WEIGHT) {
    return BikeState::Docked;
  }
  if (ticket < DOCKED_WEIGHT + RIDING_WEIGHT) {
    return BikeState::Riding;
  }
  return BikeState::InRepair;
}

FleetSnapshot makeFleet(SplitMix64& rng, const std::vector<Station>& stations, int bikes) {
  FleetSnapshot fleet;
  fleet.state.resize(bikes);
  fleet.battery.resize(bikes);
  fleet.x.resize(bikes);
  fleet.y.resize(bikes);
  fleet.dockedAt.assign(bikes, NOT_DOCKED);
  for (int i = 0; i < bikes; ++i) {
    fleet.state[i] = pickState(rng);
    fleet.battery[i] = rng.below(FULL_BATTERY_PERCENT + 1);
    fleet.x[i] = rng.below(CITY_SIDE_M);
    fleet.y[i] = rng.below(CITY_SIDE_M);
    if (fleet.state[i] == BikeState::Docked) {
      const int station = rng.below(STATIONS);
      fleet.dockedAt[i] = station;
      fleet.x[i] = stations[station].x;
      fleet.y[i] = stations[station].y;
    }
  }
  return fleet;
}

RiderRequests makeRiderRequests(SplitMix64& rng, int requests) {
  RiderRequests riders;
  riders.x.resize(requests);
  riders.y.resize(requests);
  for (int q = 0; q < requests; ++q) {
    riders.x[q] = rng.below(CITY_SIDE_M);
    riders.y[q] = rng.below(CITY_SIDE_M);
  }
  return riders;
}

CandidateLists cpuCandidateLists(const FleetSnapshot& fleet, const RiderRequests& riders) {
  const int bikes = static_cast<int>(fleet.state.size());
  const int requests = static_cast<int>(riders.x.size());
  CandidateLists lists;
  lists.start.resize(requests + 1);
  lists.bike.reserve(static_cast<size_t>(requests) * EXPECTED_CANDIDATES_PER_REQUEST);
  for (int q = 0; q < requests; ++q) {
    lists.start[q] = static_cast<int>(lists.bike.size());
    for (int i = 0; i < bikes; ++i) {
      if (isReadyBike(fleet.state[i],
                      fleet.battery[i],
                      fleet.x[i],
                      fleet.y[i],
                      riders.x[q],
                      riders.y[q],
                      SEARCH_RADIUS_M)) {
        lists.bike.push_back(i);
      }
    }
  }
  lists.start[requests] = static_cast<int>(lists.bike.size());
  return lists;
}

DeviceData uploadData(const FleetSnapshot& fleet,
                      const RiderRequests& riders,
                      const SearchGrid& grid) {
  DeviceData data;
  data.state = upload(fleet.state);
  data.battery = upload(fleet.battery);
  data.x = upload(fleet.x);
  data.y = upload(fleet.y);
  data.requestX = upload(riders.x);
  data.requestY = upload(riders.y);
  data.counts = allocateDeviceInts(grid.cells);
  data.scannedCounts = allocateDeviceInts(grid.cells);
  data.scratch = allocateDeviceInts(scratchIntsForHierarchicalScan(grid.cells));
  data.candidates = nullptr;
  return data;
}

void freeDeviceData(const DeviceData& data) {
  CHECK(cudaFree(data.state));
  int* const intBuffers[] = {data.battery,
                             data.x,
                             data.y,
                             data.requestX,
                             data.requestY,
                             data.counts,
                             data.scannedCounts,
                             data.scratch,
                             data.candidates};
  for (int* buffer : intBuffers) {
    CHECK(cudaFree(buffer));
  }
}

int countAndScanCandidates(const DeviceData& data, const SearchGrid& grid) {
  launch(count_candidates,
         sectionsByRequestsGrid(grid),
         dim3(F_THREADS),
         data.state,
         data.battery,
         data.x,
         data.y,
         data.requestX,
         data.requestY,
         data.counts,
         grid.bikes,
         SEARCH_RADIUS_M);
  scanInclusive(data.counts, data.scannedCounts, grid.cells, data.scratch);
  return copyIntToHost(data.scannedCounts + (grid.cells - 1));
}

void compactCandidates(const DeviceData& data, const SearchGrid& grid) {
  launch(compact_candidates,
         sectionsByRequestsGrid(grid),
         dim3(F_THREADS),
         data.state,
         data.battery,
         data.x,
         data.y,
         data.requestX,
         data.requestY,
         data.scannedCounts,
         data.candidates,
         grid.bikes,
         SEARCH_RADIUS_M);
}

CandidateLists downloadCandidateLists(const DeviceData& data, const SearchGrid& grid, int total) {
  CandidateLists lists;
  lists.bike = downloadInts(data.candidates, total);
  const std::vector<int> scanned = downloadInts(data.scannedCounts, grid.cells);
  lists.start.resize(grid.requests + 1);
  for (int q = 0; q < grid.requests; ++q) {
    lists.start[q] = exclusive_from_inclusive(scanned.data(), q * grid.fleetSections);
  }
  lists.start[grid.requests] = total;
  return lists;
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

long long squaredDistance(int x, int y, int qx, int qy) {
  const long long dx = x - qx;
  const long long dy = y - qy;
  return dx * dx + dy * dy;
}

int integerSqrt(long long value) {
  int root = 0;
  while (static_cast<long long>(root + 1) * (root + 1) <= value) {
    ++root;
  }
  return root;
}

struct BestBike {
  int bike;
  long long squaredDistance;
};

bool closerOrMoreCharged(const BestBike& candidate,
                         const BestBike& best,
                         const FleetSnapshot& fleet) {
  if (best.bike < 0 || candidate.squaredDistance < best.squaredDistance) {
    return true;
  }
  return candidate.squaredDistance == best.squaredDistance &&
         fleet.battery[candidate.bike] > fleet.battery[best.bike];
}

BestBike bestCandidate(const CandidateLists& lists,
                       int q,
                       const FleetSnapshot& fleet,
                       const RiderRequests& riders) {
  BestBike best = {-1, -1};
  for (int k = lists.start[q]; k < lists.start[q + 1]; ++k) {
    const int i = lists.bike[k];
    const BestBike candidate = {i,
                                squaredDistance(fleet.x[i], fleet.y[i], riders.x[q], riders.y[q])};
    if (closerOrMoreCharged(candidate, best, fleet)) {
      best = candidate;
    }
  }
  return best;
}

void printSuggestions(const CandidateLists& lists,
                      const FleetSnapshot& fleet,
                      const RiderRequests& riders) {
  const int requests = static_cast<int>(riders.x.size());
  std::printf("\nFirst rider requests of the batch (q<n> at the rider position):\n");
  for (int q = 0; q < imin(requests, REQUESTS_SHOWN); ++q) {
    const BestBike best = bestCandidate(lists, q, fleet, riders);
    std::printf("  q%-3d rider at (%4d, %4d) m: %3d bikes ready",
                q,
                riders.x[q],
                riders.y[q],
                lists.start[q + 1] - lists.start[q]);
    if (best.bike >= 0) {
      std::printf(", best b%d at station s%d, %d m away, battery %d %%\n",
                  best.bike,
                  fleet.dockedAt[best.bike],
                  integerSqrt(best.squaredDistance),
                  fleet.battery[best.bike]);
    } else {
      std::printf(" -> no charged bike within %d m (suggest the next station)\n", SEARCH_RADIUS_M);
    }
  }
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf("\nNote: CPU emulation times measure thread barriers, not a GPU.\n");
#endif
}

int argumentOr(int argc, char** argv, int position, int fallback) {
  return (argc > position) ? std::atoi(argv[position]) : fallback;
}

void printResultTable(double cpuMs, int expectedTotal, double gpuMs, int total, bool pass) {
  std::printf(
      "  %-36s %10s %8s %11s  %s\n", "version", "time [ms]", "vs CPU", "candidates", "check");
  std::printf("  %-36s %10.3f %8s %11d  %s\n",
              "CPU nested loops (reference)",
              cpuMs,
              "1.00x",
              expectedTotal,
              "-");
  std::printf("  %-36s %10.3f %7.2fx %11d  %s\n",
              "GPU count + scan + staged compact",
              gpuMs,
              cpuMs / gpuMs,
              total,
              pass ? "PASS" : "FAIL");
}

int main(int argc, char** argv) {
  const int bikes = argumentOr(argc, argv, 1, DEFAULT_BIKES);
  const int requests = argumentOr(argc, argv, 2, DEFAULT_REQUESTS);
  if (bikes < 1 || requests < 1) {
    std::fprintf(stderr, "need at least one bike and one rider request\n");
    return 1;
  }
  std::printf("Bikes ready for each rider: docked, battery >= %d %%, within %d m\n",
              MIN_BATTERY_PERCENT,
              SEARCH_RADIUS_M);
  std::printf("backend: %s, %d bikes, %d rider requests -> %lld predicate evaluations\n\n",
              CUDA_BACKEND,
              bikes,
              requests,
              static_cast<long long>(bikes) * requests);

  SplitMix64 rng(SNAPSHOT_SEED);
  const std::vector<Station> stations = makeStations(rng);
  const FleetSnapshot fleet = makeFleet(rng, stations, bikes);
  const RiderRequests riders = makeRiderRequests(rng, requests);

  CandidateLists expected;
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    expected = cpuCandidateLists(fleet, riders);
  });
  const int expectedTotal = expected.start[requests];

  const SearchGrid grid = makeSearchGrid(bikes, requests);
  DeviceData data = uploadData(fleet, riders, grid);

  const int capacity = imax(countAndScanCandidates(data, grid), 1);
  data.candidates = allocateDeviceInts(capacity);

  int total = 0;
  bool overflow = false;
  auto run = [&] {
    total = countAndScanCandidates(data, grid);
    if (total > capacity) {
      overflow = true;
      return;
    }
    compactCandidates(data, grid);
  };
  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    run();
  }
  CHECK(cudaDeviceSynchronize());
  const double gpuMs = averageMs(TIMED_RUNS, run);
  CHECK(cudaGetLastError());

  bool pass = !overflow && total == expectedTotal;
  CandidateLists lists;
  if (pass) {
    lists = downloadCandidateLists(data, grid, total);
    pass = (lists.start == expected.start) && (lists.bike == expected.bike);
  }

  printResultTable(cpuMs, expectedTotal, gpuMs, total, pass);
  if (pass) {
    printSuggestions(lists, fleet, riders);
  }
  printEmulationNote();
  std::printf("\n%s\n", pass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeDeviceData(data);
  return pass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_idle_bikes.cu -o bikeshare_idle_bikes && ./bikeshare_idle_bikes

## `bikeshare_late_returns.cu`

order-preserving filter of trips over the included time, gather into a report with fares, comparison with the unordered version

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_late_returns.cu
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#include "cuda_compat.h"

constexpr int BK_THREADS = 256;
constexpr int BK_SECTION = 2 * BK_THREADS;
constexpr int F_THREADS = 256;
constexpr int F_COARSEN = 8;
constexpr int F_SECTION = F_THREADS * F_COARSEN;

constexpr int OFFSET_THREADS = 256;
constexpr int SIMPLE_THREADS = 256;
constexpr int GATHER_THREADS = 256;

#ifndef CPU_TRIPS
#define CPU_TRIPS 60000
#endif
constexpr int DEFAULT_TRIPS = BY_BACKEND(4000000, CPU_TRIPS);
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr uint64_t TRIP_LOG_SEED = 7;

constexpr int MS_PER_SECOND = 1000;
constexpr int MS_PER_MINUTE = 60 * MS_PER_SECOND;
constexpr int MS_PER_HOUR = 60 * MS_PER_MINUTE;
constexpr int INCLUDED_MINUTES = 45;
constexpr int INCLUDED_MS = INCLUDED_MINUTES * MS_PER_MINUTE;
constexpr int EVENING_RUSH_START_MS = 16 * MS_PER_HOUR;
constexpr int EVENING_RUSH_LENGTH_MS = 4 * MS_PER_HOUR;

constexpr int PERCENT = 100;
constexpr int LEISURE_RIDE_PERCENT = 15;
constexpr int SHORTEST_COMMUTE_MS = 6 * MS_PER_MINUTE;
constexpr int COMMUTE_PARTS = 4;
constexpr int LONGEST_COMMUTE_PART_MS = 10 * MS_PER_MINUTE;
constexpr int SHORTEST_LEISURE_MS = 30 * MS_PER_MINUTE;
constexpr int LONGEST_LEISURE_MS = 150 * MS_PER_MINUTE;

constexpr int BASE_FARE_CENTS = 300;
constexpr int FARE_PER_STARTED_BLOCK_CENTS = 100;
constexpr int FARE_BLOCK_MS = 10 * MS_PER_MINUTE;
constexpr int CENTS_PER_BOLIVIANO = 100;

constexpr int STATIONS = 48;
constexpr int BIKES = 1500;
constexpr int REPORT_LINES_SHOWN = 6;

__host__ __device__ inline bool isLate(int startMs, int endMs, int includedMs) {
  return endMs - startMs > includedMs;
}

__host__ __device__ inline int fareCents(int rideMs) {
  const int startedBlocks = (rideMs + FARE_BLOCK_MS - 1) / FARE_BLOCK_MS;
  return BASE_FARE_CENTS + FARE_PER_STARTED_BLOCK_CENTS * startedBlocks;
}

__global__ void brent_kung_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
  }

  __syncthreads();
  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
  if (t == 0) {
    sums[blockIdx.x] = XY[BK_SECTION - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__device__ void kogge_stone_in_shared(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < F_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ int count_own_flags(const int* flag, int first) {
  int count = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    count += flag[first + k];
  }
  return count;
}

__device__ void pack_own_flagged_indices(
    const int* flag, int first, int base, int pos, int* stage) {
  for (int k = 0; k < F_COARSEN; ++k) {
    if (flag[first + k]) {
      stage[pos++] = base + first + k;
    }
  }
}

__device__ void copy_out_coalesced(const int* stage, int count, int* out) {
  for (int k = threadIdx.x; k < count; k += F_THREADS) {
    out[k] = stage[k];
  }
}

__global__ void count_late(
    const int* startMs, const int* endMs, int* counts, int n, int includedMs) {
  __shared__ int blockCount;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  int c = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    const int i = blockIdx.x * F_SECTION + k * F_THREADS + threadIdx.x;
    if (i < n && isLate(startMs[i], endMs[i], includedMs)) {
      ++c;
    }
  }
  atomicAdd(&blockCount, c);
  __syncthreads();

  if (threadIdx.x == 0) {
    counts[blockIdx.x] = blockCount;
  }
}

__global__ void compact_late(const int* startMs,
                             const int* endMs,
                             const int* scannedCounts,
                             int* lateIdx,
                             int n,
                             int includedMs) {
  __shared__ int late[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  const int t = threadIdx.x;
  const int base = blockIdx.x * F_SECTION;
  const int first = t * F_COARSEN;

  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = k * F_THREADS + t;
    const int i = base + j;
    late[j] = (i < n && isLate(startMs[i], endMs[i], includedMs)) ? 1 : 0;
  }
  __syncthreads();

  part[t] = count_own_flags(late, first);
  kogge_stone_in_shared(part);
  __syncthreads();

  pack_own_flagged_indices(late, first, base, exclusive_from_inclusive(part, t), stage);
  __syncthreads();

  const int blockCount = part[F_THREADS - 1];
  const int start = exclusive_from_inclusive(scannedCounts, blockIdx.x);
  copy_out_coalesced(stage, blockCount, lateIdx + start);
}

__global__ void gather_report(const int* lateIdx,
                              int count,
                              const int* tripId,
                              const int* bikeId,
                              const int* startStation,
                              const int* startMs,
                              const int* endMs,
                              int* repTrip,
                              int* repBike,
                              int* repStation,
                              int* repEnd,
                              int* repDuration,
                              int* repFare) {
  const int k = blockIdx.x * blockDim.x + threadIdx.x;
  if (k < count) {
    const int i = lateIdx[k];
    const int duration = endMs[i] - startMs[i];
    repTrip[k] = tripId[i];
    repBike[k] = bikeId[i];
    repStation[k] = startStation[i];
    repEnd[k] = endMs[i];
    repDuration[k] = duration;
    repFare[k] = fareCents(duration);
  }
}

__global__ void late_by_atomics(
    const int* startMs, const int* endMs, int* lateIdx, int* counter, int n, int includedMs) {
  __shared__ int blockCount;
  __shared__ int blockStart;
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const bool k = i < n && isLate(startMs[i], endMs[i], includedMs);
  int local = 0;
  if (k) {
    local = atomicAdd(&blockCount, 1);
  }
  __syncthreads();

  if (threadIdx.x == 0) {
    blockStart = atomicAdd(counter, blockCount);
  }
  __syncthreads();

  if (k) {
    lateIdx[blockStart + local] = i;
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int bound) {
    return static_cast<int>(next() % static_cast<uint64_t>(bound));
  }

  int between(int lowest, int highest) {
    return lowest + below(highest - lowest + 1);
  }

  bool percentChance(int percent) {
    return below(PERCENT) < percent;
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

struct Trip {
  int tripId;
  int bikeId;
  int startStation;
  int startMs;
  int endMs;
};

bool dockedEarlier(const Trip& a, const Trip& b) {
  if (a.endMs != b.endMs) {
    return a.endMs < b.endMs;
  }
  return a.tripId < b.tripId;
}

struct TripLog {
  std::vector<int> tripId;
  std::vector<int> bikeId;
  std::vector<int> startStation;
  std::vector<int> startMs;
  std::vector<int> endMs;
};

struct ReportTable {
  std::vector<int> lateIndex;
  std::vector<int> trip;
  std::vector<int> bike;
  std::vector<int> station;
  std::vector<int> endMs;
  std::vector<int> duration;
  std::vector<int> fare;
};

struct DeviceLog {
  int* tripId;
  int* bikeId;
  int* startStation;
  int* startMs;
  int* endMs;
};

struct DeviceReport {
  int* trip;
  int* bike;
  int* station;
  int* endMs;
  int* duration;
  int* fare;
};

struct DeviceWork {
  int* sectionCounts;
  int* scannedSectionCounts;
  int* scratch;
  int* lateIndex;
  int* counter;
};

int blocksFor(int n, int blockSize) {
  return static_cast<int>(cdiv(n, blockSize));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

int* uploadInts(const std::vector<int>& values) {
  int* pointer = allocateDeviceInts(static_cast<int>(values.size()));
  CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(int), cudaMemcpyHostToDevice));
  return pointer;
}

std::vector<int> downloadInts(const int* source, int count) {
  std::vector<int> values(count);
  if (count > 0) {
    CHECK(cudaMemcpy(values.data(), source, count * sizeof(int), cudaMemcpyDeviceToHost));
  }
  return values;
}

int copyIntToHost(const int* source) {
  int value = 0;
  CHECK(cudaMemcpy(&value, source, sizeof(int), cudaMemcpyDeviceToHost));
  return value;
}

int scratchIntsForHierarchicalScan(int n) {
  const int blocks = blocksFor(n, BK_SECTION);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks);
}

void scanInclusive(const int* d_in, int* d_out, int n, int* scratch) {
  const int blocks = blocksFor(n, BK_SECTION);
  int* blockSums = scratch;
  launch(brent_kung_scan, dim3(blocks), dim3(BK_THREADS), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanInclusive(blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(add_block_offsets,
           dim3(blocks),
           dim3(OFFSET_THREADS),
           d_out,
           scannedBlockSums,
           n,
           BK_SECTION);
  }
}

int syntheticRideMs(SplitMix64& rng) {
  if (rng.percentChance(LEISURE_RIDE_PERCENT)) {
    return rng.between(SHORTEST_LEISURE_MS, LONGEST_LEISURE_MS);
  }
  int rideMs = SHORTEST_COMMUTE_MS;
  for (int part = 0; part < COMMUTE_PARTS; ++part) {
    rideMs += rng.below(LONGEST_COMMUTE_PART_MS + 1);
  }
  return rideMs;
}

TripLog makeTripLog(int trips) {
  SplitMix64 rng(TRIP_LOG_SEED);
  std::vector<Trip> rows(trips);
  for (int i = 0; i < trips; ++i) {
    const int startMs = EVENING_RUSH_START_MS + rng.below(EVENING_RUSH_LENGTH_MS);
    const int endMs = startMs + syntheticRideMs(rng);
    rows[i] = {i, rng.below(BIKES), rng.below(STATIONS), startMs, endMs};
  }
  std::sort(rows.begin(), rows.end(), dockedEarlier);

  TripLog log;
  for (const Trip& row : rows) {
    log.tripId.push_back(row.tripId);
    log.bikeId.push_back(row.bikeId);
    log.startStation.push_back(row.startStation);
    log.startMs.push_back(row.startMs);
    log.endMs.push_back(row.endMs);
  }
  return log;
}

std::vector<int> cpuLateIndices(const TripLog& log) {
  std::vector<int> lateIndices;
  lateIndices.reserve(log.startMs.size());
  for (size_t i = 0; i < log.startMs.size(); ++i) {
    if (isLate(log.startMs[i], log.endMs[i], INCLUDED_MS)) {
      lateIndices.push_back(static_cast<int>(i));
    }
  }
  return lateIndices;
}

DeviceLog uploadLog(const TripLog& log) {
  DeviceLog device;
  device.tripId = uploadInts(log.tripId);
  device.bikeId = uploadInts(log.bikeId);
  device.startStation = uploadInts(log.startStation);
  device.startMs = uploadInts(log.startMs);
  device.endMs = uploadInts(log.endMs);
  return device;
}

DeviceReport allocateDeviceReport(int trips) {
  DeviceReport report;
  report.trip = allocateDeviceInts(trips);
  report.bike = allocateDeviceInts(trips);
  report.station = allocateDeviceInts(trips);
  report.endMs = allocateDeviceInts(trips);
  report.duration = allocateDeviceInts(trips);
  report.fare = allocateDeviceInts(trips);
  return report;
}

DeviceWork allocateDeviceWork(int trips) {
  const int sections = blocksFor(trips, F_SECTION);
  DeviceWork work;
  work.sectionCounts = allocateDeviceInts(sections);
  work.scannedSectionCounts = allocateDeviceInts(sections);
  work.scratch = allocateDeviceInts(scratchIntsForHierarchicalScan(sections));
  work.lateIndex = allocateDeviceInts(trips);
  work.counter = allocateDeviceInts(1);
  return work;
}

void freeDevice(const DeviceLog& log, const DeviceReport& report, const DeviceWork& work) {
  int* const buffers[] = {log.tripId,
                          log.bikeId,
                          log.startStation,
                          log.startMs,
                          log.endMs,
                          work.sectionCounts,
                          work.scannedSectionCounts,
                          work.scratch,
                          work.lateIndex,
                          work.counter,
                          report.trip,
                          report.bike,
                          report.station,
                          report.endMs,
                          report.duration,
                          report.fare};
  for (int* buffer : buffers) {
    CHECK(cudaFree(buffer));
  }
}

int compactLateInLogSequence(const DeviceLog& log, const DeviceWork& work, int trips) {
  const int sections = blocksFor(trips, F_SECTION);
  launch(count_late,
         dim3(sections),
         dim3(F_THREADS),
         log.startMs,
         log.endMs,
         work.sectionCounts,
         trips,
         INCLUDED_MS);
  scanInclusive(work.sectionCounts, work.scannedSectionCounts, sections, work.scratch);
  launch(compact_late,
         dim3(sections),
         dim3(F_THREADS),
         log.startMs,
         log.endMs,
         work.scannedSectionCounts,
         work.lateIndex,
         trips,
         INCLUDED_MS);
  return copyIntToHost(work.scannedSectionCounts + (sections - 1));
}

void gatherReport(const DeviceLog& log,
                  const DeviceWork& work,
                  const DeviceReport& report,
                  int lateCount) {
  launch(gather_report,
         dim3(blocksFor(imax(lateCount, 1), GATHER_THREADS)),
         dim3(GATHER_THREADS),
         work.lateIndex,
         lateCount,
         log.tripId,
         log.bikeId,
         log.startStation,
         log.startMs,
         log.endMs,
         report.trip,
         report.bike,
         report.station,
         report.endMs,
         report.duration,
         report.fare);
}

int compactLateWithAtomics(const DeviceLog& log, const DeviceWork& work, int trips) {
  CHECK(cudaMemset(work.counter, 0, sizeof(int)));
  launch(late_by_atomics,
         dim3(blocksFor(trips, SIMPLE_THREADS)),
         dim3(SIMPLE_THREADS),
         log.startMs,
         log.endMs,
         work.lateIndex,
         work.counter,
         trips,
         INCLUDED_MS);
  return copyIntToHost(work.counter);
}

ReportTable downloadReport(const DeviceReport& report, const DeviceWork& work, int lateCount) {
  ReportTable table;
  table.lateIndex = downloadInts(work.lateIndex, lateCount);
  table.trip = downloadInts(report.trip, lateCount);
  table.bike = downloadInts(report.bike, lateCount);
  table.station = downloadInts(report.station, lateCount);
  table.endMs = downloadInts(report.endMs, lateCount);
  table.duration = downloadInts(report.duration, lateCount);
  table.fare = downloadInts(report.fare, lateCount);
  return table;
}

bool rowMatchesLog(const ReportTable& table, int k, int i, const TripLog& log) {
  const int duration = log.endMs[i] - log.startMs[i];
  return table.lateIndex[k] == i && table.trip[k] == log.tripId[i] &&
         table.bike[k] == log.bikeId[i] && table.station[k] == log.startStation[i] &&
         table.endMs[k] == log.endMs[i] && table.duration[k] == duration &&
         table.fare[k] == fareCents(duration);
}

bool reportMatchesLog(const ReportTable& table,
                      const std::vector<int>& expectedIndices,
                      const TripLog& log) {
  for (size_t k = 0; k < table.lateIndex.size(); ++k) {
    if (!rowMatchesLog(table, static_cast<int>(k), expectedIndices[k], log)) {
      return false;
    }
  }
  return true;
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

template <class Work>
void warmUp(Work work) {
  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    work();
  }
}

template <class Work>
double timeRuns(Work work) {
  CHECK(cudaDeviceSynchronize());
  const double ms = averageMs(TIMED_RUNS, work);
  CHECK(cudaGetLastError());
  return ms;
}

void fillWithMinusOne(int* d_values, int count) {
  CHECK(cudaMemset(d_values, 0xff, count * sizeof(int)));
}

const char* yesNo(bool value) {
  return value ? "yes" : "no";
}

const char* verdict(bool pass) {
  return pass ? "PASS" : "FAIL";
}

int indexOfLargest(const std::vector<int>& values) {
  return static_cast<int>(std::max_element(values.begin(), values.end()) - values.begin());
}

void printClock(int ms) {
  std::printf("%02d:%02d:%02d", ms / MS_PER_HOUR, ms / MS_PER_MINUTE % 60, ms / MS_PER_SECOND % 60);
}

void printReportLine(const ReportTable& table, int k) {
  std::printf(
      "  trip t%-6d bike b%-5d from s%-3d docked ", table.trip[k], table.bike[k], table.station[k]);
  printClock(table.endMs[k]);
  std::printf("  ride %3d min  fare Bs %d.%02d\n",
              table.duration[k] / MS_PER_MINUTE,
              table.fare[k] / CENTS_PER_BOLIVIANO,
              table.fare[k] % CENTS_PER_BOLIVIANO);
}

void printWorstStationAndBike(const ReportTable& table) {
  std::vector<int> lateByStation(STATIONS, 0);
  std::vector<int> lateByBike(BIKES, 0);
  for (size_t k = 0; k < table.trip.size(); ++k) {
    ++lateByStation[table.station[k]];
    ++lateByBike[table.bike[k]];
  }
  const int worstStation = indexOfLargest(lateByStation);
  const int worstBike = indexOfLargest(lateByBike);
  std::printf("  most late returns: start station s%d (%d), bike b%d (%d)\n",
              worstStation,
              lateByStation[worstStation],
              worstBike,
              lateByBike[worstBike]);
}

void printReport(const ReportTable& table, int trips) {
  const int lateCount = static_cast<int>(table.trip.size());
  std::printf("\n%d of %d trips over the included %d min (%.1f %%). First lines, by dock time:\n",
              lateCount,
              trips,
              INCLUDED_MINUTES,
              100.0 * lateCount / trips);
  for (int k = 0; k < imin(lateCount, REPORT_LINES_SHOWN); ++k) {
    printReportLine(table, k);
  }
  printWorstStationAndBike(table);
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf("\nNote: CPU emulation times measure thread barriers, not a GPU.\n");
#endif
}

int tripCount(int argc, char** argv) {
  return (argc > 1) ? std::atoi(argv[1]) : DEFAULT_TRIPS;
}

void printResultTable(double cpuMs,
                      int expectedCount,
                      double sequenceMs,
                      int lateCount,
                      bool sequencePass,
                      double atomicMs,
                      int atomicCount,
                      bool atomicInLogSequence,
                      bool atomicPass) {
  std::printf("  %-43s %10s %8s %8s %9s  %s\n",
              "version",
              "time [ms]",
              "vs CPU",
              "late",
              "by time",
              "check");
  std::printf("  %-43s %10.3f %8s %8d %9s  %s\n",
              "CPU loop (reference)",
              cpuMs,
              "1.00x",
              expectedCount,
              "yes",
              "-");
  std::printf("  %-43s %10.3f %7.2fx %8d %9s  %s\n",
              "GPU count + scan + staged compact + gather",
              sequenceMs,
              cpuMs / sequenceMs,
              lateCount,
              yesNo(sequencePass),
              verdict(sequencePass));
  std::printf("  %-43s %10.3f %7.2fx %8d %9s  %s\n",
              "GPU block-private atomics (no gather)",
              atomicMs,
              cpuMs / atomicMs,
              atomicCount,
              yesNo(atomicInLogSequence),
              verdict(atomicPass));
}

int main(int argc, char** argv) {
  const int trips = tripCount(argc, argv);
  if (trips < 1) {
    std::fprintf(stderr, "need at least one trip\n");
    return 1;
  }
  std::printf("Late returns report (sequence-preserving filter + gather)\n");
  std::printf("backend: %s, %d finished trips, late = more than %d min\n\n",
              CUDA_BACKEND,
              trips,
              INCLUDED_MINUTES);

  const TripLog log = makeTripLog(trips);
  std::vector<int> expectedIndices;
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    expectedIndices = cpuLateIndices(log);
  });
  const int expectedCount = static_cast<int>(expectedIndices.size());

  const DeviceLog deviceLog = uploadLog(log);
  const DeviceReport deviceReport = allocateDeviceReport(trips);
  const DeviceWork work = allocateDeviceWork(trips);

  int lateCount = 0;
  auto runInSequence = [&] {
    lateCount = compactLateInLogSequence(deviceLog, work, trips);
    gatherReport(deviceLog, work, deviceReport, lateCount);
  };
  warmUp(runInSequence);
  fillWithMinusOne(work.lateIndex, trips);
  const double sequenceMs = timeRuns(runInSequence);

  bool sequencePass = (lateCount == expectedCount);
  ReportTable table;
  if (sequencePass && lateCount > 0) {
    table = downloadReport(deviceReport, work, lateCount);
    sequencePass = reportMatchesLog(table, expectedIndices, log);
  }

  int atomicCount = 0;
  auto runWithAtomics = [&] {
    atomicCount = compactLateWithAtomics(deviceLog, work, trips);
  };
  warmUp(runWithAtomics);
  const double atomicMs = timeRuns(runWithAtomics);
  bool atomicPass = (atomicCount == expectedCount);
  bool atomicInLogSequence = false;
  if (atomicPass && atomicCount > 0) {
    std::vector<int> found = downloadInts(work.lateIndex, atomicCount);
    atomicInLogSequence = (found == expectedIndices);
    std::sort(found.begin(), found.end());
    atomicPass = (found == expectedIndices);
  }

  printResultTable(cpuMs,
                   expectedCount,
                   sequenceMs,
                   lateCount,
                   sequencePass,
                   atomicMs,
                   atomicCount,
                   atomicInLogSequence,
                   atomicPass);

  if (sequencePass && lateCount > 0) {
    printReport(table, trips);
  }
  printEmulationNote();

  const bool allPass = sequencePass && atomicPass;
  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeDevice(deviceLog, deviceReport, work);
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_late_returns.cu -o bikeshare_late_returns && ./bikeshare_late_returns

## `filter.cu`

the five variants (global atomic, block-private, flags + scan + scatter, count + scan + staged compact, single pass) at 10 % and 50 % selectivity, with a CPU reference, order and multiset checks, and timings

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile filter.cu
#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <random>
#include <vector>

#include "cuda_compat.h"

constexpr int SIMPLE_THREADS = 256;
constexpr int BK_THREADS = 256;
constexpr int BK_SECTION = 2 * BK_THREADS;
constexpr int F_THREADS = 256;
constexpr int F_COARSEN = 8;
constexpr int F_SECTION = F_THREADS * F_COARSEN;

constexpr int OFFSET_THREADS = 256;
constexpr int FLAG_READY = 1;

constexpr int DEFAULT_ELEMENTS = BY_BACKEND(16000003, 65537);
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr unsigned INPUT_SEED = 2025;
constexpr int VALUE_RANGE = 1000;
constexpr int THRESHOLD_KEEPING_ABOUT_10_PERCENT = 100;
constexpr int THRESHOLD_KEEPING_ABOUT_50_PERCENT = 500;
constexpr int VALUES_SHOWN = 10;

__host__ __device__ inline bool keep(int v, int threshold) {
  return v < threshold;
}

__global__ void filter_global_atomic(const int* in, int* out, int* counter, int n, int threshold) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    const int v = in[i];
    if (keep(v, threshold)) {
      out[atomicAdd(counter, 1)] = v;
    }
  }
}

__global__ void filter_block_private(const int* in, int* out, int* counter, int n, int threshold) {
  __shared__ int blockCount;
  __shared__ int blockStart;
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const int v = (i < n) ? in[i] : 0;
  const bool k = (i < n) && keep(v, threshold);
  int local = 0;
  if (k) {
    local = atomicAdd(&blockCount, 1);
  }
  __syncthreads();

  if (threadIdx.x == 0) {
    blockStart = atomicAdd(counter, blockCount);
  }
  __syncthreads();

  if (k) {
    out[blockStart + local] = v;
  }
}

__global__ void brent_kung_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
  }

  __syncthreads();
  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
  if (t == 0) {
    sums[blockIdx.x] = XY[BK_SECTION - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__global__ void make_flags(const int* in, int* flags, int n, int threshold) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    flags[i] = keep(in[i], threshold) ? 1 : 0;
  }
}

__global__ void scatter_kept(const int* in, const int* flags, const int* pos, int* out, int n) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n && flags[i]) {
    out[pos[i] - 1] = in[i];
  }
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__device__ void load_section_coalesced(const int* in, int n, int base, int* vals) {
  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = k * F_THREADS + threadIdx.x;
    vals[j] = (base + j < n) ? in[base + j] : 0;
  }
}

__device__ bool is_real_and_kept(const int* vals, int j, int base, int n, int threshold) {
  return base + j < n && keep(vals[j], threshold);
}

__device__ void kogge_stone_in_shared(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < F_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void copy_out_coalesced(const int* stage, int count, int* out) {
  for (int k = threadIdx.x; k < count; k += F_THREADS) {
    out[k] = stage[k];
  }
}

__device__ int block_compact(
    const int* in, int n, int base, int threshold, int* vals, int* stage, int* part) {
  const int t = threadIdx.x;
  const int first = t * F_COARSEN;

  load_section_coalesced(in, n, base, vals);
  __syncthreads();

  int count = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      ++count;
    }
  }
  part[t] = count;

  kogge_stone_in_shared(part);
  __syncthreads();

  int pos = exclusive_from_inclusive(part, t);
  for (int k = 0; k < F_COARSEN; ++k) {
    if (is_real_and_kept(vals, first + k, base, n, threshold)) {
      stage[pos++] = vals[first + k];
    }
  }
  __syncthreads();
  return part[F_THREADS - 1];
}

__global__ void count_kept(const int* in, int* counts, int n, int threshold) {
  __shared__ int blockCount;
  if (threadIdx.x == 0) {
    blockCount = 0;
  }
  __syncthreads();

  const int base = blockIdx.x * F_SECTION;
  int c = 0;
  for (int k = 0; k < F_COARSEN; ++k) {
    const int j = base + k * F_THREADS + threadIdx.x;
    if (j < n && keep(in[j], threshold)) {
      ++c;
    }
  }
  atomicAdd(&blockCount, c);
  __syncthreads();

  if (threadIdx.x == 0) {
    counts[blockIdx.x] = blockCount;
  }
}

__global__ void compact_staged(
    const int* in, int* out, const int* scannedCounts, int n, int threshold) {
  __shared__ int vals[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  const int count = block_compact(in, n, blockIdx.x * F_SECTION, threshold, vals, stage, part);
  const int start = exclusive_from_inclusive(scannedCounts, blockIdx.x);
  copy_out_coalesced(stage, count, out + start);
}

__device__ int next_ticket(int* ticket) {
  return atomicAdd(ticket, 1);
}

__device__ int atomic_read(int* address) {
  return atomicAdd(address, 0);
}

__device__ void wait_until_ready(int* flag) {
  int state = atomic_read(flag);
  while (state != FLAG_READY) {
    state = atomic_read(flag);
  }
}

__device__ int wait_for_sum_of_previous_sections(int* flags, int* prefix, int bid) {
  if (bid == 0) {
    return 0;
  }
  wait_until_ready(&flags[bid]);
  return atomic_read(&prefix[bid]);
}

__device__ void publish_sum_of_sections_up_to(int bid, int sum, int* flags, int* prefix) {
  atomicExch(&prefix[bid + 1], sum);
  __threadfence();
  atomicExch(&flags[bid + 1], FLAG_READY);
}

__global__ void compact_single_pass(
    const int* in, int* out, int* flags, int* prefix, int* ticket, int n, int threshold) {
  __shared__ int vals[F_SECTION];
  __shared__ int stage[F_SECTION];
  __shared__ int part[F_THREADS];
  __shared__ int bid_s;
  __shared__ int start_s;

  if (threadIdx.x == 0) {
    bid_s = next_ticket(ticket);
  }
  __syncthreads();
  const int bid = bid_s;

  const int count = block_compact(in, n, bid * F_SECTION, threshold, vals, stage, part);

  if (threadIdx.x == 0) {
    const int before = wait_for_sum_of_previous_sections(flags, prefix, bid);
    publish_sum_of_sections_up_to(bid, before + count, flags, prefix);
    start_s = before;
  }
  __syncthreads();

  copy_out_coalesced(stage, count, out + start_s);
}

enum class FilterVariant {
  GlobalAtomic,
  BlockPrivate,
  FlagsScanScatter,
  CountScanCompact,
  SinglePass
};

struct VariantInfo {
  FilterVariant variant;
  const char* name;
  bool preservesOrder;
};

const VariantInfo FILTER_VARIANTS[] = {
    {FilterVariant::GlobalAtomic, "1 global atomic per kept element", false},
    {FilterVariant::BlockPrivate, "2 block-private counter", false},
    {FilterVariant::FlagsScanScatter, "3 flags + scan + scatter", true},
    {FilterVariant::CountScanCompact, "4 count + scan + staged compact", true},
    {FilterVariant::SinglePass, "5 single pass (domino)", true},
};

struct Buffers {
  int* input;
  int* output;
  int* counter;
  int* flags;
  int* positions;
  int* scratch;
  int* sectionCounts;
  int* scannedSectionCounts;
  int* chainFlags;
  int* chainPrefix;
  int* ticket;
};

int blocksFor(int n, int blockSize) {
  return static_cast<int>(cdiv(n, blockSize));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

int copyIntToHost(const int* source) {
  int value = 0;
  CHECK(cudaMemcpy(&value, source, sizeof(int), cudaMemcpyDeviceToHost));
  return value;
}

int scratchIntsForHierarchicalScan(int n) {
  const int blocks = blocksFor(n, BK_SECTION);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks);
}

void scanInclusive(const int* d_in, int* d_out, int n, int* scratch) {
  const int blocks = blocksFor(n, BK_SECTION);
  int* blockSums = scratch;
  launch(brent_kung_scan, dim3(blocks), dim3(BK_THREADS), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanInclusive(blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(add_block_offsets,
           dim3(blocks),
           dim3(OFFSET_THREADS),
           d_out,
           scannedBlockSums,
           n,
           BK_SECTION);
  }
}

Buffers allocateBuffers(int n) {
  const int sections = blocksFor(n, F_SECTION);
  Buffers buffers;
  buffers.input = allocateDeviceInts(n);
  buffers.output = allocateDeviceInts(n);
  buffers.counter = allocateDeviceInts(1);
  buffers.flags = allocateDeviceInts(n);
  buffers.positions = allocateDeviceInts(n);
  buffers.scratch = allocateDeviceInts(scratchIntsForHierarchicalScan(n));
  buffers.sectionCounts = allocateDeviceInts(sections);
  buffers.scannedSectionCounts = allocateDeviceInts(sections);
  buffers.chainFlags = allocateDeviceInts(sections + 1);
  buffers.chainPrefix = allocateDeviceInts(sections + 1);
  buffers.ticket = allocateDeviceInts(1);
  return buffers;
}

void freeBuffers(const Buffers& buffers) {
  CHECK(cudaFree(buffers.input));
  CHECK(cudaFree(buffers.output));
  CHECK(cudaFree(buffers.counter));
  CHECK(cudaFree(buffers.flags));
  CHECK(cudaFree(buffers.positions));
  CHECK(cudaFree(buffers.scratch));
  CHECK(cudaFree(buffers.sectionCounts));
  CHECK(cudaFree(buffers.scannedSectionCounts));
  CHECK(cudaFree(buffers.chainFlags));
  CHECK(cudaFree(buffers.chainPrefix));
  CHECK(cudaFree(buffers.ticket));
}

int filterWithGlobalAtomic(const Buffers& buffers, int n, int threshold) {
  CHECK(cudaMemset(buffers.counter, 0, sizeof(int)));
  launch(filter_global_atomic,
         dim3(blocksFor(n, SIMPLE_THREADS)),
         dim3(SIMPLE_THREADS),
         buffers.input,
         buffers.output,
         buffers.counter,
         n,
         threshold);
  return copyIntToHost(buffers.counter);
}

int filterWithBlockPrivateCounter(const Buffers& buffers, int n, int threshold) {
  CHECK(cudaMemset(buffers.counter, 0, sizeof(int)));
  launch(filter_block_private,
         dim3(blocksFor(n, SIMPLE_THREADS)),
         dim3(SIMPLE_THREADS),
         buffers.input,
         buffers.output,
         buffers.counter,
         n,
         threshold);
  return copyIntToHost(buffers.counter);
}

int filterWithFlagsScanScatter(const Buffers& buffers, int n, int threshold) {
  const int blocks = blocksFor(n, SIMPLE_THREADS);
  launch(
      make_flags, dim3(blocks), dim3(SIMPLE_THREADS), buffers.input, buffers.flags, n, threshold);
  scanInclusive(buffers.flags, buffers.positions, n, buffers.scratch);
  launch(scatter_kept,
         dim3(blocks),
         dim3(SIMPLE_THREADS),
         buffers.input,
         buffers.flags,
         buffers.positions,
         buffers.output,
         n);
  return copyIntToHost(buffers.positions + (n - 1));
}

int filterWithCountScanCompact(const Buffers& buffers, int n, int threshold) {
  const int sections = blocksFor(n, F_SECTION);
  launch(count_kept,
         dim3(sections),
         dim3(F_THREADS),
         buffers.input,
         buffers.sectionCounts,
         n,
         threshold);
  scanInclusive(buffers.sectionCounts, buffers.scannedSectionCounts, sections, buffers.scratch);
  launch(compact_staged,
         dim3(sections),
         dim3(F_THREADS),
         buffers.input,
         buffers.output,
         buffers.scannedSectionCounts,
         n,
         threshold);
  return copyIntToHost(buffers.scannedSectionCounts + (sections - 1));
}

void resetDominoChain(const Buffers& buffers, int sections) {
  const int chainEntries = sections + 1;
  CHECK(cudaMemset(buffers.chainFlags, 0, chainEntries * sizeof(int)));
  CHECK(cudaMemset(buffers.chainPrefix, 0, chainEntries * sizeof(int)));
  CHECK(cudaMemset(buffers.ticket, 0, sizeof(int)));
}

int filterInSinglePass(const Buffers& buffers, int n, int threshold) {
  const int sections = blocksFor(n, F_SECTION);
  resetDominoChain(buffers, sections);
  launch(compact_single_pass,
         dim3(sections),
         dim3(F_THREADS),
         buffers.input,
         buffers.output,
         buffers.chainFlags,
         buffers.chainPrefix,
         buffers.ticket,
         n,
         threshold);
  return copyIntToHost(buffers.chainPrefix + sections);
}

int runVariant(FilterVariant variant, const Buffers& buffers, int n, int threshold) {
  int keptCount = 0;
  switch (variant) {
    case FilterVariant::GlobalAtomic:
      keptCount = filterWithGlobalAtomic(buffers, n, threshold);
      break;
    case FilterVariant::BlockPrivate:
      keptCount = filterWithBlockPrivateCounter(buffers, n, threshold);
      break;
    case FilterVariant::FlagsScanScatter:
      keptCount = filterWithFlagsScanScatter(buffers, n, threshold);
      break;
    case FilterVariant::CountScanCompact:
      keptCount = filterWithCountScanCompact(buffers, n, threshold);
      break;
    case FilterVariant::SinglePass:
      keptCount = filterInSinglePass(buffers, n, threshold);
      break;
  }
  CHECK(cudaGetLastError());
  return keptCount;
}

std::vector<int> randomValues(int n) {
  std::mt19937 rng(INPUT_SEED);
  std::uniform_int_distribution<int> value(0, VALUE_RANGE - 1);
  std::vector<int> values(n);
  for (int& v : values) {
    v = value(rng);
  }
  return values;
}

std::vector<int> cpuFilter(const std::vector<int>& input, int threshold) {
  std::vector<int> kept;
  kept.reserve(input.size());
  for (int v : input) {
    if (keep(v, threshold)) {
      kept.push_back(v);
    }
  }
  return kept;
}

std::vector<int> sorted(std::vector<int> values) {
  std::sort(values.begin(), values.end());
  return values;
}

void fillWithMinusOne(int* d_values, int count) {
  CHECK(cudaMemset(d_values, 0xff, count * sizeof(int)));
}

std::vector<int> readOutput(const Buffers& buffers, int count) {
  std::vector<int> output(count);
  CHECK(cudaMemcpy(output.data(), buffers.output, count * sizeof(int), cudaMemcpyDeviceToHost));
  return output;
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

const char* yesNo(bool value) {
  return value ? "yes" : "no";
}

struct Expected {
  std::vector<int> inInputOrder;
  std::vector<int> sortedValues;
};

bool measureAndCheckVariant(const VariantInfo& info,
                            const Buffers& buffers,
                            int n,
                            int threshold,
                            const Expected& expected,
                            double cpuMs) {
  auto run = [&] {
    return runVariant(info.variant, buffers, n, threshold);
  };
  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    run();
  }
  fillWithMinusOne(buffers.output, n);
  CHECK(cudaDeviceSynchronize());

  int keptCount = 0;
  const double ms = averageMs(TIMED_RUNS, [&] {
    keptCount = run();
  });

  const int expectedCount = static_cast<int>(expected.inInputOrder.size());
  bool pass = (keptCount == expectedCount);
  bool inInputOrder = false;
  if (pass) {
    const std::vector<int> output = readOutput(buffers, keptCount);
    inInputOrder = (output == expected.inInputOrder);
    pass = info.preservesOrder ? inInputOrder : (sorted(output) == expected.sortedValues);
  }

  std::printf("  %-34s %10.3f %7.2fx %8d %9s  %s\n",
              info.name,
              ms,
              cpuMs / ms,
              keptCount,
              yesNo(inInputOrder),
              pass ? "PASS" : "FAIL");
  return pass;
}

bool runAtThreshold(const std::vector<int>& input, const Buffers& buffers, int threshold) {
  const int n = static_cast<int>(input.size());
  Expected expected;
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    expected.inInputOrder = cpuFilter(input, threshold);
  });
  expected.sortedValues = sorted(expected.inInputOrder);
  const int expectedCount = static_cast<int>(expected.inInputOrder.size());

  std::printf("threshold %d: %d of %d kept (%.1f %%)\n",
              threshold,
              expectedCount,
              n,
              100.0 * expectedCount / n);
  std::printf("  %-34s %10s %8s %8s %9s  %s\n",
              "variant",
              "time [ms]",
              "vs CPU",
              "kept",
              "in order",
              "check");
  std::printf("  %-34s %10.3f %8s %8d %9s  %s\n",
              "CPU sequential loop (reference)",
              cpuMs,
              "1.00x",
              expectedCount,
              "yes",
              "-");

  bool allPass = true;
  for (const VariantInfo& info : FILTER_VARIANTS) {
    const bool pass = measureAndCheckVariant(info, buffers, n, threshold, expected, cpuMs);
    allPass = allPass && pass;
  }
  std::printf("\n");
  return allPass;
}

void printValues(const std::vector<int>& values) {
  for (int v : values) {
    std::printf("%4d", v);
  }
}

void showWhatUnorderedMeans(const std::vector<int>& input, const Buffers& buffers) {
  const int n = static_cast<int>(input.size());
  const int threshold = THRESHOLD_KEEPING_ABOUT_10_PERCENT;
  std::vector<int> firstInInputOrder = cpuFilter(input, threshold);
  firstInInputOrder.resize(imin(static_cast<int>(firstInInputOrder.size()), VALUES_SHOWN));

  const int keptCount = runVariant(FilterVariant::GlobalAtomic, buffers, n, threshold);
  const std::vector<int> firstFromAtomic = readOutput(buffers, imin(keptCount, VALUES_SHOWN));

  std::printf("First kept values, threshold %d:\n  input order       ", threshold);
  printValues(firstInInputOrder);
  std::printf("\n  global atomic     ");
  printValues(firstFromAtomic);
  std::printf("\n  (variants 3-5 always reproduce the input order; 1-2 only by chance)\n");
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf(
      "\nNote: CPU emulation times measure thread barriers, not a GPU; run on a GPU for real "
      "timings.\n");
#endif
}

int elementCount(int argc, char** argv) {
  return (argc > 1) ? std::atoi(argv[1]) : DEFAULT_ELEMENTS;
}

int main(int argc, char** argv) {
  const int n = elementCount(argc, argv);
  if (n < 1) {
    std::fprintf(stderr, "n must be >= 1\n");
    return 1;
  }
  std::printf("Filtering / stream compaction (chapter 12)  backend: %s\n", CUDA_BACKEND);
  std::printf("n = %d ints uniform in [0, %d), keep value < threshold, %d timed run(s) each\n\n",
              n,
              VALUE_RANGE,
              TIMED_RUNS);

  const std::vector<int> input = randomValues(n);
  const Buffers buffers = allocateBuffers(n);
  CHECK(cudaMemcpy(buffers.input, input.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  bool allPass = true;
  const int thresholds[] = {THRESHOLD_KEEPING_ABOUT_10_PERCENT, THRESHOLD_KEEPING_ABOUT_50_PERCENT};
  for (int threshold : thresholds) {
    const bool pass = runAtThreshold(input, buffers, threshold);
    allPass = allPass && pass;
  }

  showWhatUnorderedMeans(input, buffers);
  printEmulationNote();
  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeBuffers(buffers);
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. filter.cu -o filter && ./filter

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/10_filter/README.md`.

**Next:** open `ec2/11_merge/colab_1_step_by_step.ipynb`.